# Efektywne programowanie w języku Python

## Efektywnie 1

### Od part01 do pomiarów

W `part01` poznaliśmy składnię i znaczenie operacji. Tutaj użyjemy ich do sprawdzenia, co wpływa na czytelność, czas i pamięć.

**Kolejność pracy:** najpierw poprawność i koszt algorytmu, potem profilowanie i możliwy zysk całego programu, a dopiero na końcu mikrobenchmark pojedynczej operacji.

Pierwsze dwa bloki nadają się do pracy po podstawach i kolekcjach. Profilowanie, wpływ lokalnej poprawki oraz pomiary pamięci można wykorzystać ponownie na późniejszych zajęciach o wydajności. Kod obsługujący narzędzia pomiarowe jest przygotowany do demonstracji; nie musisz jeszcze umieć go samodzielnie odtworzyć.

### Wspólna procedura: od poprawnego kodu do pomiaru

**Określ wymagania → sprawdź wynik i skutki uboczne → zmierz → zmień rozwiązanie → ponownie sprawdź poprawność → porównaj czas i pamięć.**

Przed porównaniem ustal dopuszczalne dane, wynik, zachowanie dla pustych lub brakujących danych i to, czy funkcja może zmieniać przekazane obiekty. Sama zgodność wyniku dla jednego przykładu nie wystarcza.

In [1]:
# assumptions
N = 1_000_000

## 1. Najpierw zrozum problem

Efektywność może wynikać z usunięcia zbędnej pracy, doboru algorytmu lub użycia gotowego narzędzia. Krótszy zapis sam w sobie nie dowodzi większej szybkości.

### a) Suma liczb całkowitych od 1 do N

Zakładamy, że `N` jest nieujemną liczbą całkowitą. Wzór wykorzystuje strukturę tego konkretnego problemu; nie zastępuje sumowania dowolnych danych.

In [2]:
total = 0
for x in range(1, N + 1):
    total += x
print(total)

500000500000


In [3]:
formula_total = N * (N + 1) // 2
print(formula_total)
assert total == sum(range(1, N + 1)) == formula_total
assert sum(range(1, 0 + 1)) == 0 * (0 + 1) // 2 == 0

500000500000


### Jak uczciwie mierzyć?

- Najpierw sprawdź zgodność wyników i przypadki brzegowe.
- Porównuj te same dane, zakres pracy oraz środowisko.
- Przygotuj dane poza pomiarem, jeśli mierzysz samą operację. Jeśli ważny jest koszt całego rozwiązania, zmierz go osobno.
- `%%timeit` wykonuje kod wielokrotnie; bez `-n` dobiera liczbę wykonań automatycznie. Podsumowanie opisuje zmienność pomiarów, nie gwarantowany czas na każdym komputerze.
- Notuj wersję Pythona. Czas i zużycie pamięci to osobne kryteria; `timeit` nie mierzy pamięci.

`%%timeit` jest magią IPythona, nie zwykłą składnią Pythona. [Dokumentacja IPythona](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-timeit).

In [ ]:
import sys
print(sys.version)
print(sys.executable)

In [4]:
%%timeit
total = 0
for x in range(1, N + 1):
    total += x

148 ms ± 3.29 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
%%timeit
sum(range(1, N + 1))

105 ms ± 19.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
%%timeit
N * (N + 1) // 2

325 ns ± 5.86 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


### Co rzeczywiście przyspieszyliśmy?

Pętla i `sum(range(...))` odwiedzają kolejne liczby. Wzór nie wykonuje tej iteracji. W modelu operacji o stałym koszcie porównujemy O(N) z O(1); dla bardzo dużych liczb koszt arytmetyki całkowitej zależy także od liczby bitów.

Nie nazywaj zmiennej `sum`: zasłonisz funkcję wbudowaną i kolejne `sum(...)` zgłosi `TypeError`.

### b) Długość listy — oddziel utworzenie danych od operacji

Lista przechowuje swoją długość: `len(elements)` nie przegląda wszystkich elementów. Ręczne liczenie wykonuje O(N) kroków, a `len()` listy jest O(1). Samo `[0] * N` wymaga utworzenia listy.

Generator zwykle nie ma `len()`; policzenie jego elementów wymaga przejścia po nim i zużywa generator.

In [7]:
elements = [0] * N
size = 0
for element in elements:
    size += 1
print(size)

1000000


In [ ]:
print(len(elements))
assert size == len(elements) == N

In [ ]:
# Osobno sprawdzamy pustą listę.
assert len([]) == 0

In [ ]:
%%timeit
size = 0
for element in elements:
    size += 1

In [ ]:
%%timeit
len(elements)

### Osobny pomiar tworzenia listy

Ten pomiar obejmuje przydzielenie listy i pobranie jej długości. Odpowiada na inne pytanie niż samo `len(elements)`.

In [ ]:
%%timeit
len([0] * N)

### Eksperyment: jak czas zmienia się wraz z N?

Porównaj sumowanie pętlą, funkcję wbudowaną i wzór dla kilku rozmiarów. Wyniki wypisujemy poza mierzonym kodem; `autorange()` dobiera liczbę wykonań, a `repeat()` pozwala zobaczyć zmienność.

Nie oczekuj idealnych proporcji dla małych danych: narzut wywołania i szum pomiaru również mają znaczenie.

**Przygotowane narzędzie demonstracyjne:** kod poniżej organizuje serię pomiarów. Na tym etapie skup się na zgodności danych, liczbie powtórzeń i wynikach. `lambda` oraz domyślne argumenty zachowują wybraną metodę i rozmiar danych dla pomiaru; szczegóły funkcji jako obiektów omówimy później.

In [ ]:
# Przygotowane narzędzie: obsługa serii pomiarów.
import timeit
from statistics import median

def sum_loop(n):
    total = 0
    for value in range(1, n + 1):
        total += value
    return total

def sum_builtin(n):
    return sum(range(1, n + 1))



In [ ]:
def sum_formula(n):
    return n * (n + 1) // 2

methods = {"pętla": sum_loop, "sum": sum_builtin, "wzór": sum_formula}
for n in (10, 1_000, 100_000):
    expected = sum_formula(n)
    for label, method in methods.items():
        assert method(n) == expected
        timer = timeit.Timer(lambda method=method, n=n: method(n))
        loops, _ = timer.autorange()
        samples = [elapsed / loops for elapsed in timer.repeat(repeat=3, number=loops)]
        print(f"N={n:>6}, {label:>5}: mediana {median(samples) * 1e6:.3f} µs; "
              f"zakres {min(samples) * 1e6:.3f}–{max(samples) * 1e6:.3f} µs")

## 2. Powtórka idiomów — nowy cel

Zamianę wartości i operator `in` znamy z `part01`. Tutaj porównujemy warianty pod względem czytelności i liczby pomocniczych kroków. Sam krótszy zapis nie jest dowodem większej szybkości.

### a) Zamiana wartości

In [ ]:
# Wariant z tymczasową zmienną — poprawny, ale dłuższy.
x = 10
y = -10
print(f"Before: x = {x}, y = {y}")
tmp = x
x = y
y = tmp
print(f"After: x = {x}, y = {y}")
assert (x, y) == (-10, 10)

In [ ]:
# Idiom Pythona: prawa strona jest obliczana przed przypisaniem.
x, y = 10, -10
print(f"Before: x = {x}, y = {y}")
x, y = y, x
print(f"After: x = {x}, y = {y}")
assert (x, y) == (-10, 10)

In [ ]:
# Celowy błąd: ValueError — 9 nazw, ale 10 wartości do rozpakowania.
# Przewidź wynik i odczytaj komunikat wyjątku.
a, b, c, d, e, f, g, h, i = 1021, 32, 42, 232, 12, 323, 22, 41, 6, 10

### b) Wyszukiwanie i for/else

`else` pętli wykonuje się po jej normalnym zakończeniu bez `break`, także dla pustej kolekcji. `return` i wyjątek również przerywają wykonanie przed `else`.

[Dokumentacja for/else](https://docs.python.org/3/tutorial/controlflow.html#break-and-continue-statements-and-else-clauses-on-loops).

In [ ]:
needle = 'd'
letters = ['a', 'b', 'g', 'p']

In [ ]:
# Wariant z flagą — poprawny.
found = False
for letter in letters:
    if needle == letter:
        print('Found')
        found = True
        break
if not found:
    print('Not found!')

In [ ]:
# Wariant for/else — bez pomocniczej flagi.
for letter in letters:
    if needle == letter:
        print('Found')
        break
else: # If no break occurred
    print('Not found!')

### Jeśli pytasz tylko o obecność, użyj in

`for/else` przydaje się, gdy podczas wyszukiwania wykonujesz dodatkowe czynności lub masz bardziej złożony warunek. Do samego sprawdzenia obecności elementu wystarcza operator `in`.

In [ ]:
print("Found" if needle in letters else "Not found!")

def find_with_flag(needle, values):
    found = False
    for value in values:
        if value == needle:
            found = True
            break
    return found



In [ ]:
def find_with_loop_else(needle, values):
    for value in values:
        if value == needle:
            break
    else:
        return False
    return True

for candidate, values in (("a", letters), ("d", letters), ("a", [])):
    expected = candidate in values
    assert find_with_flag(candidate, values) == expected
    assert find_with_loop_else(candidate, values) == expected
    print(f"{candidate!r} w {values}: {expected}")

### Przygotowanie: pętla, lista składana i generator

Trzy zapisy obliczają kwadraty tych samych liczb. Pętla z `append()` i **list comprehension** tworzą listę wyników. **Wyrażenie generatorowe** w nawiasach okrągłych oblicza kolejne wartości dopiero podczas iteracji.

Generator jest jednorazowy. `sum()` przechodzi po nim do końca; następne przejście nie dostanie tych samych wartości. `list(generator)` też go zużywa i dodatkowo tworzy listę.

To minimum do dalszych demonstracji. Pełne omówienie comprehensions jest w `part02`, a iteratorów i generatorów w `part03`.

In [ ]:
squares_by_loop = []
for value in range(5):
    squares_by_loop.append(value * value)

squares_by_list = [value * value for value in range(5)]
squares_once = (value * value for value in range(5))
assert squares_by_loop == squares_by_list == [0, 1, 4, 9, 16]
print(squares_by_list)
print(sum(squares_once))   # 30: generator zostaje zużyty.
print(list(squares_once))  # []: nie zaczyna ponownie od początku.
assert list(squares_once) == []
assert sum(value * value for value in range(5)) == sum(squares_by_list)
# Nowe wyrażenie tworzy nowy generator do kolejnego obliczenia.

## 3. Najpierw znajdź kosztowne miejsce — profilowanie (10 minut)

W większym programie zacznij od pytania: **gdzie znika czas?** Profilowanie pokazuje liczbę wywołań i czas w poszczególnych funkcjach. `cProfile` jest częścią biblioteki standardowej.

Przykład ma trzy etapy: przygotowanie danych, obliczenie sumy kwadratów i przygotowanie podsumowania. Przewidź, który etap będzie najdroższy, a potem odczytaj raport.

Profiler dodaje narzut: używamy go do szukania kosztownych miejsc. Wybrane rozwiązania porównujemy osobno przez `timeit`. [Dokumentacja cProfile](https://docs.python.org/3/library/profile.html).

Obsługa profilera poniżej jest przygotowaną demonstracją. Funkcje opisujące trzy etapy programu czytaj jak zwykły kod z `part01`; `runcall()` i `pstats.Stats` służą tutaj do zebrania i pokazania raportu.

In [ ]:
import cProfile
import pstats

def prepare_values(n):
    return list(range(n))

def calculate_checksum(values):
    total = 0
    for value in values:
        total += value * value
    return total



In [ ]:
def summarize_values(values, checksum):
    return {"count": len(values), "maximum": max(values, default=None),
            "checksum": checksum}

def analyze_values(n):
    values = prepare_values(n)
    checksum = calculate_checksum(values)
    return summarize_values(values, checksum)

In [ ]:
profile_n = 100_000
profiler = cProfile.Profile()
profile_result = profiler.runcall(analyze_values, profile_n)
assert profile_result["checksum"] == sum(x * x for x in range(profile_n))
pstats.Stats(profiler).strip_dirs().sort_stats("cumulative").print_stats(8)


In [ ]:
print(profile_result)

### Jak czytać raport?

- `ncalls`: liczba wywołań funkcji.
- `tottime`: czas w samej funkcji, bez wywoływanych przez nią funkcji.
- `cumtime`: czas funkcji wraz z jej wywołaniami.

`analyze_values` obejmuje wszystkie etapy. **Nie sumuj czasów `cumtime` rodzica i dzieci** — policzyłbyś tę samą pracę wielokrotnie.

Wskaż etap do poprawy na podstawie swojego raportu. Zaproponuj zamiennik, sprawdź wynik i zmierz cały program przed zmianą oraz po niej.

## 4. Ile zyska cały program? (5 minut)

Załóżmy, że program trwa 10 sekund. Poprawiany fragment zajmuje 1 sekundę, a pozostałe etapy 9 sekund. Dwukrotne przyspieszenie fragmentu daje **9 + 0,5 = 9,5 sekundy**: czas całego programu skrócił się o 5%.

Dla udziału `p` poprawianego fragmentu i jego przyspieszenia `s`:

- nowy czas / stary czas: `(1 - p) + p / s`;
- przyspieszenie całego programu: `1 / ((1 - p) + p / s)`.

To prosty model: zakładamy ten sam wynik, taki sam zakres pracy i niezmieniony koszt pozostałych etapów. Udział `p` ustal na podstawie pomiarów.

In [ ]:
from math import isclose

def overall_time_ratio(fraction, local_speedup):
    if not 0 <= fraction <= 1 or local_speedup <= 0:
        raise ValueError("Udział musi być w [0, 1], a przyspieszenie dodatnie")
    return (1 - fraction) + fraction / local_speedup

time_ratio = overall_time_ratio(0.10, 2)
assert isclose(time_ratio, 0.95)
print(f"Nowy czas: {10 * time_ratio:.2f} s")
print(f"Skrócenie czasu: {(1 - time_ratio) * 100:.1f}%")
print(f"Przyspieszenie całego programu: {1 / time_ratio:.3f}×")

### Przewidź efekt

1. Fragment zajmuje 50% czasu i przyspieszamy go dwukrotnie. O ile skróci się cały czas?
2. Fragment zajmuje 10% czasu. Jakie jest maksymalne skrócenie czasu, nawet gdy koszt tego fragmentu stanie się pomijalny?
3. Czy duże przyspieszenie pojedynczej operacji `x*x` musi być widoczne w czasie całego programu?

## 5. Mikrobenchmark: x**2 kontra x*x

Porównujemy operacje na małej liczbie całkowitej. Nie wyciągaj ogólnego wniosku dla innych typów, wykładników ani wersji interpretera. `x**2` dobrze wyraża potęgowanie; zmianę uzasadniaj pomiarem w rzeczywistym problemie.

Poniższe pomiary obejmują również pętlę. Osobny pomiar pustej pętli pokaże skalę narzutu, ale nie odejmuj mechanicznie dwóch zaszumionych wyników. Najpierw sprawdź zgodność wartości.

In [8]:
for value in (-10, 0, 5, 10**20):
    assert value**2 == value * value

In [9]:
%%timeit
for i in range(10_000):
    pass

519 µs ± 11 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [12]:
%%timeit
x = 5
for i in range(10_000):
    y = x**10.0

5.05 ms ± 668 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [13]:
%%timeit
x = 5
for i in range(10_000):
    y = x * x * x * x * x * x * x * x * x * x 

6.65 ms ± 83.3 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## 6. Czas i pamięć to różne koszty (10 minut)

Porównamy sumowanie kwadratów z użyciem listy i generatora:

```python
sum([x * x for x in range(n)])
sum(x * x for x in range(n))
```

Lista materializuje wszystkie kwadraty przed sumowaniem. Generator przekazuje je kolejno. Wynik jest taki sam, ale nie zakładaj z góry, który wariant będzie szybszy.

### Co mierzy tracemalloc?

`tracemalloc` śledzi przydziały pamięci obsługiwane przez alokatory Pythona. Odczytamy **szczyt pamięci od rozpoczęcia śledzenia**, osobno dla każdego wariantu. To nie jest pomiar całej pamięci procesu ani wszystkich alokacji bibliotek natywnych. [Dokumentacja tracemalloc](https://docs.python.org/3/library/tracemalloc.html).

Funkcja `measure_peak_memory` jest przygotowanym narzędziem demonstracyjnym. Przyjmuje inną funkcję jako argument; `try/finally` zapewnia wyłączenie śledzenia także po błędzie. Szczegóły przekazywania funkcji i obsługi wyjątków wrócą na dalszych zajęciach. Teraz interpretujemy mierzoną operację i szczyt pamięci.

In [3]:
# Przygotowane narzędzie: pomiar pamięci osobno dla każdego wariantu.
import tracemalloc

def squares_list(n):
    return sum([x * x for x in range(n)])

def squares_generator(n):
    return sum(x * x for x in range(n))



In [4]:
def measure_peak_memory(function, n):
    if tracemalloc.is_tracing():
        raise RuntimeError("Zakończ wcześniejszy pomiar tracemalloc przed tym przykładem")
    tracemalloc.start()
    try:
        result = function(n)
        _, peak = tracemalloc.get_traced_memory()
    finally:
        tracemalloc.stop()
    return result, peak

memory_n = 100_000
list_result, list_peak = measure_peak_memory(squares_list, memory_n)
generator_result, generator_peak = measure_peak_memory(squares_generator, memory_n)


In [5]:
assert list_result == generator_result == memory_n * (memory_n - 1) * (2 * memory_n - 1) // 6
print(f"Lista:     szczyt {list_peak / 1024:.1f} KiB")
print(f"Generator: szczyt {generator_peak / 1024:.1f} KiB")

Lista:     szczyt 3779.7 KiB
Generator: szczyt 1.1 KiB


### Zmierz czas osobno

Śledzenie pamięci jest już wyłączone. Porównaj czas obu funkcji dla tych samych danych. Narzut `tracemalloc` nie powinien wpływać na benchmark czasu.

Generator jest jednorazowy: w każdym powtórzeniu tworzymy nowy wewnątrz funkcji. Zużyty generator nie wykona ponownie tej samej pracy.

In [ ]:
%%timeit
squares_list(memory_n)

In [ ]:
%%timeit
squares_generator(memory_n)

## 7. Ćwiczenie: wybierz rozwiązanie i uzasadnij (5 minut)

Zadanie: policz sumę kwadratów liczb od `0` do `n - 1`. Masz trzy warianty: `calculate_checksum(range(n))`, `squares_list(n)` i `squares_generator(n)`.

1. **Przewidź** kolejność według czasu oraz pamięci. Zapisz przewidywanie przed pomiarem.
2. Sprawdź zgodność wyników dla `n = 0`, `1`, `10` i `100_000`.
3. Zmierz czas i szczyt pamięci osobno. Dla czasu porównaj co najmniej `n = 10` oraz `100_000`; zanotuj wersję Pythona.
4. Wybierz wariant do programu, w którym pamięć jest ograniczona. Uzasadnij wybór wynikiem, zużyciem pamięci i czytelnością.

Czy wyniki potwierdzają Twoje przewidywanie? Które wnioski dotyczą algorytmu, a które tylko badanego środowiska?

**Dla chętnych:** wykorzystaj wzór na sumę kwadratów. Sprawdź zakres, przypadek `n = 0` i poprawność wyniku przed pomiarem.

## Podsumowanie — przewidź i wyjaśnij

1. Która zmiana usuwa iterację, która używa gotowego narzędzia, a która tylko skraca zapis?
2. Dlaczego `len([0] * N)` i `len(elements)` mierzą różny zakres pracy?
3. Kiedy wykona się `else` pętli dla pustej listy i dla znalezionego elementu?
4. Czy wynik `x*x` kontra `x**2` na jednym interpreterze uzasadnia zmianę każdego potęgowania w projekcie?
5. Czym różni się profilowanie programu od benchmarku wybranej funkcji?
6. Dlaczego mniejsze zużycie pamięci nie musi oznaczać krótszego czasu wykonania?


## Źródła i materiały dodatkowe

- [Profilowanie: cProfile i pstats](https://docs.python.org/3/library/profile.html).
- [Pomiar pamięci: tracemalloc](https://docs.python.org/3/library/tracemalloc.html).
- [Dokumentacja timeit](https://docs.python.org/3/library/timeit.html).
- [Magia timeit w IPythonie](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-timeit).
- [Pętle for/else](https://docs.python.org/3/tutorial/controlflow.html#break-and-continue-statements-and-else-clauses-on-loops).
- [Sebastian Witowski — Writing faster Python](https://www.youtube.com/watch?v=YjHsOrOOSuI).
- [7 Simple Tricks to Write Better Python Code](https://www.youtube.com/watch?v=VBokjWj_cEA).

Linki sprawdzone 2026-10-06: dokumentacja jest dostępna, a oba filmy otwierają się i odtwarzają bez logowania. Filmy są materiałami sprzed około 10 lat; wyniki pomiarów i zalecenia dotyczące szybkości należy sprawdzać w używanej wersji Pythona. Dokumentacja stanowi podstawowe źródło do przykładów.